# РГР «От восприятия среды к движению робота»

Стартовая тетрадь версии **0.7.0**. Математическая работа разбита на четыре блока: **поле риска → маршруты → векторное поле → траектория**.

A* и симулятор здесь уже реализованы: их нужно использовать как вычислительные инструменты, а не переписывать с нуля.

## 0. Загрузка JSON варианта

In [ ]:
import json, math, heapq, sys
from copy import deepcopy
from pathlib import Path

# Для своей работы оставьте USE_DEMO_VARIANT=False.
USE_DEMO_VARIANT = False
SCENE_FILE = "variant.json"  # используется только при локальном запуске вне Colab

if USE_DEMO_VARIANT:
    scene = json.load(open("example_variant_000.json", encoding="utf-8"))
else:
    try:
        from google.colab import files
        print("Выберите JSON, скачанный из генератора варианта.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("JSON не загружен.")
        name, data = next(iter(uploaded.items()))
        scene = json.loads(data.decode("utf-8"))
        print("Загружен:", name)
    except ImportError:
        p = Path(SCENE_FILE)
        if not p.exists():
            raise FileNotFoundError(
                "Укажите имя JSON вашей пары в SCENE_FILE или запустите тетрадь в Google Colab."
            )
        scene = json.load(open(p, encoding="utf-8"))

CFG = scene["solver_config"]
PACKAGE_VERSION=CFG["package_version"]; SCHEMA_VERSION=CFG["schema_version"]
WORKSPACE_W=CFG["workspace_width"]; WORKSPACE_H=CFG["workspace_height"]
ROBOT_RADIUS=CFG["robot_radius"]; RISK_BACKGROUND=CFG["risk_background"]
UNCERTAINTY_HARD_FACTOR=CFG["uncertainty_hard_factor"]
ASTAR_GRID_STEP=CFG["astar_grid_step"]; ASTAR_MARGIN=CFG["astar_margin"]
QUAD_H=CFG["quadrature_h"]; QUAD_REL_TOL=CFG["quadrature_rel_tol"]
DT=CFG["dt"]; VMAX=CFG["vmax"]; WAYPOINT_STRIDE=CFG["waypoint_stride"]
WAYPOINT_TOL=CFG["waypoint_tol"]; GOAL_TOL=CFG["goal_tol"]; MAX_STEPS=CFG["max_steps"]
TRAJECTORY_RECORD_STRIDE=CFG["trajectory_record_stride"]
RDP_EPS=0.16

print("Вариант", scene["variant"], "| версия модели", scene["package_version"])


## Служебное вычислительное ядро
Ниже находится каноническая реализация A*, интеграла по маршруту и симуляции. В обычной работе этот блок **не требуется редактировать**.

In [ ]:
def round_half_up(x):
    return int(math.floor(x + 0.5))


def risk(x, y, obs, s0=RISK_BACKGROUND):
    v = s0
    for o in obs:
        a = o['radius'] + o['sigma']
        dx, dy = x-o['x'], y-o['y']
        v += o['confidence'] * a*a / (dx*dx + dy*dy + a*a)
    return v


def grad_risk(x, y, obs):
    gx = gy = 0.0
    for o in obs:
        a = o['radius'] + o['sigma']
        dx, dy = x-o['x'], y-o['y']
        q = dx*dx + dy*dy + a*a
        c = -2.0 * o['confidence'] * a*a / (q*q)
        gx += c*dx; gy += c*dy
    return gx, gy


def laplacian_risk(x, y, obs):
    z = 0.0
    for o in obs:
        a = o['radius'] + o['sigma']
        dx, dy = x-o['x'], y-o['y']
        r2 = dx*dx + dy*dy
        z += 4.0*o['confidence']*a*a*(r2-a*a)/(r2+a*a)**3
    return z


def model_forbidden_radius(o, robot_r=ROBOT_RADIUS):
    return o['radius'] + robot_r + UNCERTAINTY_HARD_FACTOR*o['sigma']


def point_model_clearance(x, y, obs, robot_r=ROBOT_RADIUS):
    return min(math.hypot(x-o['x'], y-o['y']) - model_forbidden_radius(o, robot_r) for o in obs)


def blocked_point(x, y, obs, extra=0.0, robot_r=ROBOT_RADIUS):
    # center of a disk robot must also stay inside the rectangular workspace
    if x < robot_r+extra or x > WORKSPACE_W-robot_r-extra or y < robot_r+extra or y > WORKSPACE_H-robot_r-extra:
        return True
    for o in obs:
        rr = model_forbidden_radius(o, robot_r) + extra
        if (x-o['x'])**2 + (y-o['y'])**2 <= rr*rr:
            return True
    return False


def point_segment_distance(c, a, b):
    cx, cy = c; ax, ay = a; bx, by = b
    vx, vy = bx-ax, by-ay
    vv = vx*vx + vy*vy
    if vv <= 1e-15:
        return math.hypot(cx-ax, cy-ay)
    t = ((cx-ax)*vx + (cy-ay)*vy)/vv
    t = max(0.0, min(1.0, t))
    px, py = ax+t*vx, ay+t*vy
    return math.hypot(cx-px, cy-py)


def segment_clearance(a, b, obs, robot_r=ROBOT_RADIUS):
    """Obstacle clearance only, matching d_min in the assignment."""
    best = float('inf')
    for o in obs:
        d = point_segment_distance((o['x'],o['y']), a, b) - model_forbidden_radius(o, robot_r)
        best = min(best, d)
    return best


def workspace_segment_clearance(a, b, robot_r=ROBOT_RADIUS):
    """Clearance to the rectangular workspace boundary, reported separately from d_min."""
    best = float('inf')
    for x,y in (a,b):
        best=min(best, x-robot_r, WORKSPACE_W-robot_r-x, y-robot_r, WORKSPACE_H-robot_r-y)
    return best


def segment_blocked(a, b, obs, extra=0.0, robot_r=ROBOT_RADIUS):
    # exact circle/segment check + conservative workspace check (segment of rectangle convex)
    for p in (a,b):
        if p[0] < robot_r+extra or p[0] > WORKSPACE_W-robot_r-extra or p[1] < robot_r+extra or p[1] > WORKSPACE_H-robot_r-extra:
            return True
    for o in obs:
        rr = model_forbidden_radius(o, robot_r) + extra
        if point_segment_distance((o['x'],o['y']), a, b) <= rr:
            return True
    return False


def astar(start, goal, obs, lambda_J, grid_step=ASTAR_GRID_STEP, margin=ASTAR_MARGIN):
    nx = round_half_up(WORKSPACE_W/grid_step)+1
    ny = round_half_up(WORKSPACE_H/grid_step)+1
    def idx(p):
        return (max(0,min(nx-1,round_half_up(p[0]/grid_step))), max(0,min(ny-1,round_half_up(p[1]/grid_step))))
    def pt(u): return (u[0]*grid_step, u[1]*grid_step)
    s, g = idx(start), idx(goal)
    if blocked_point(*pt(s), obs, margin) or blocked_point(*pt(g), obs, margin): return None
    moves=[(1,0,1.0),(0,1,1.0),(-1,0,1.0),(0,-1,1.0),(1,1,math.sqrt(2)),(-1,1,math.sqrt(2)),(1,-1,math.sqrt(2)),(-1,-1,math.sqrt(2))]
    gc={s:0.0}; prev={}; pq=[(math.dist(pt(s),pt(g)),0.0,s[0],s[1],s)]
    closed=set()
    while pq:
        fcur,gcur,_,_,u=heapq.heappop(pq)
        if u in closed: continue
        closed.add(u)
        if u==g: break
        p0=pt(u)
        for di,dj,mult in moves:
            v=(u[0]+di,u[1]+dj)
            if not(0<=v[0]<nx and 0<=v[1]<ny): continue
            p1=pt(v)
            if segment_blocked(p0,p1,obs,margin): continue
            edge_len=grid_step*mult
            rv=.5*(risk(*p0,obs)+risk(*p1,obs))
            ng=gcur+edge_len*(1.0+lambda_J*rv)
            if ng < gc.get(v,float('inf'))-1e-12:
                gc[v]=ng; prev[v]=u
                h=math.dist(p1,pt(g))
                heapq.heappush(pq,(ng+h,ng,v[0],v[1],v))
    if g not in gc: return None
    nodes=[];u=g
    while True:
        nodes.append(u)
        if u==s: break
        u=prev[u]
    nodes.reverse()
    return [pt(u) for u in nodes]


def route_length(P):
    return sum(math.dist(a,b) for a,b in zip(P,P[1:]))


def route_integral(P, obs, h):
    total=0.0
    for a,b in zip(P,P[1:]):
        L=math.dist(a,b)
        n=max(1, math.ceil(L/h))
        vals=[]
        for j in range(n+1):
            t=j/n
            x=a[0]+t*(b[0]-a[0]); y=a[1]+t*(b[1]-a[1])
            vals.append(risk(x,y,obs))
        total += L*(vals[0]+vals[-1]+2*sum(vals[1:-1]))/(2*n)
    return total


def route_integral_control(P, obs, h=QUAD_H, rel_tol=QUAD_REL_TOL):
    I_h=route_integral(P,obs,h); I_h2=route_integral(P,obs,h/2)
    rel=abs(I_h2-I_h)/max(abs(I_h2),1e-12)
    return I_h,I_h2,rel,rel<=rel_tol


def route_model_clearance(P, obs, robot_r=ROBOT_RADIUS):
    if len(P)<2:return float('nan')
    return min(segment_clearance(a,b,obs,robot_r) for a,b in zip(P,P[1:]))


def route_workspace_clearance(P, robot_r=ROBOT_RADIUS):
    if len(P)<2:return float('nan')
    return min(workspace_segment_clearance(a,b,robot_r) for a,b in zip(P,P[1:]))


def route_valid(P, obs, margin=0.0):
    return all(not segment_blocked(a,b,obs,margin) for a,b in zip(P,P[1:]))


def route_metrics(P, obs, lambda_J):
    Ih,Ih2,rel,ok=route_integral_control(P,obs)
    L=route_length(P)
    return {'length':L,'risk':Ih2,'cost':L+lambda_J*Ih2,'clearance':route_model_clearance(P,obs),'workspace_clearance':route_workspace_clearance(P),'quad_I_h':Ih,'quad_I_h2':Ih2,'quad_rel':rel,'quad_ok':ok}


def choose_waypoints(path):
    wp=[path[0]]+path[WAYPOINT_STRIDE::WAYPOINT_STRIDE]
    if wp[-1]!=path[-1]:wp.append(path[-1])
    # remove accidental duplicate
    out=[]
    for p in wp:
        if not out or p!=out[-1]:out.append(p)
    return out


def rotation_sign(path,key):
    # classify by nearest route point to key center
    q=min(path,key=lambda z:(z[0]-key['x'])**2+(z[1]-key['y'])**2)
    return -1.0 if q[1]>=key['y'] else 1.0


def field(x,y,T,obs,k,beta_s,omega,key):
    gx,gy=grad_risk(x,y,obs)
    fx=k*(T[0]-x)-beta_s*gx
    fy=k*(T[1]-y)-beta_s*gy
    a=key['radius']+key['sigma']
    dx,dy=x-key['x'],y-key['y'];q=dx*dx+dy*dy+a*a
    fx += omega*(-dy)/q; fy += omega*dx/q
    return fx,fy


def simulate(start,goal,path,obs,k,beta_s,omega,key,record_stride=TRAJECTORY_RECORD_STRIDE):
    wp=choose_waypoints(path)
    x,y=start; wi=1 if len(wp)>1 else 0
    traj=[(x,y)]; minc=point_model_clearance(x,y,obs); collided=False; travelled=0.0
    def append_terminal():
        if not traj or math.dist(traj[-1], (x,y)) > 1e-12:
            traj.append((x,y))
    for it in range(MAX_STEPS):
        if math.hypot(x-goal[0],y-goal[1])<GOAL_TOL:
            append_terminal()
            return {'trajectory':traj,'trajectory_length':travelled,'reached':True,'collided':False,'steps':it,'min_clearance':minc,'waypoints':wp}
        if wi < len(wp)-1 and math.hypot(x-wp[wi][0], y-wp[wi][1]) < WAYPOINT_TOL:
            wi+=1
        T=wp[wi]
        fx,fy=field(x,y,T,obs,k,beta_s,omega,key)
        sp=math.hypot(fx,fy)
        if sp>VMAX:
            fx*=VMAX/sp; fy*=VMAX/sp
        xn,yn=x+DT*fx,y+DT*fy
        if segment_blocked((x,y),(xn,yn),obs,0.0):
            collided=True
            append_terminal()
            return {'trajectory':traj,'trajectory_length':travelled,'reached':False,'collided':True,'steps':it,'min_clearance':minc,'waypoints':wp}
        minc=min(minc,segment_clearance((x,y),(xn,yn),obs))
        travelled += math.hypot(xn-x,yn-y)
        x,y=xn,yn
        if it%record_stride==0:traj.append((x,y))
    append_terminal()
    return {'trajectory':traj,'trajectory_length':travelled,'reached':False,'collided':collided,'steps':MAX_STEPS,'min_clearance':minc,'waypoints':wp}


## 1. Поле риска
В аналитической части вы вручную выводите $M_*$, $\bar s_*$ и $\nabla s$. Этот блок нужен только для проверки сцены и визуализации.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

obs=scene["obstacles"]; start=tuple(scene["start"]); goal=tuple(scene["goal"])
key=obs[scene["key_obstacle"]]
xs=np.linspace(0,WORKSPACE_W,260); ys=np.linspace(0,WORKSPACE_H,180)
Z=np.array([[risk(x,y,obs) for x in xs] for y in ys])
fig,ax=plt.subplots(figsize=(10,6))
im=ax.imshow(Z,origin="lower",extent=[0,WORKSPACE_W,0,WORKSPACE_H],aspect="auto")
fig.colorbar(im,ax=ax,label="модельный риск s(x,y)")
for o in obs:
    ax.add_patch(Circle((o["x"],o["y"]),o["radius"],fill=False,lw=2))
    ax.add_patch(Circle((o["x"],o["y"]),model_forbidden_radius(o),fill=False,ls="--",lw=1.2))
ax.scatter(*start,label="S"); ax.scatter(*goal,marker="*",s=100,label="G")
ax.set(xlim=(0,WORKSPACE_W),ylim=(0,WORKSPACE_H),xlabel="x, м",ylabel="y, м")
ax.set_aspect("equal"); ax.legend(); plt.show()

## 2. Риск вдоль маршрута
A* используется как готовый инструмент. В отчёте нужны метрики $L$, $I$, $J$, $d_{\min}$ и математическая интерпретация результата.

In [ ]:
path=astar(start,goal,obs,scene["lambda_J"])
assert path is not None
metrics=route_metrics(path,obs,scene["lambda_J"])
omega=rotation_sign(path,key)*scene["omega_abs"]
print("A*:", metrics)
print("знак и значение omega =", omega, "м^2/с")

### Эксперимент с $p$ и $\sigma$
Участник A исследует $p$, участник B — $\sigma$.

In [ ]:
def experiment(fieldname, values):
    rows=[]
    for val in values:
        sc=deepcopy(scene); sc["obstacles"][sc["key_obstacle"]][fieldname]=val
        ob=sc["obstacles"]
        pth=astar(tuple(sc["start"]),tuple(sc["goal"]),ob,sc["lambda_J"])
        m=route_metrics(pth,ob,sc["lambda_J"])
        side="сверху" if rotation_sign(pth,ob[sc["key_obstacle"]])<0 else "снизу"
        rows.append((val,m["length"],m["risk"],m["cost"],m["clearance"],side))
    return rows

print("p:")
for row in experiment("confidence",scene["experiment"]["confidence_values"]): print(row)
print("sigma:")
for row in experiment("sigma",scene["experiment"]["sigma_values"]): print(row)

## 3. Векторное поле
Формулы для $\nabla s$, $\operatorname{div}\mathbf F$, $\operatorname{rot}\mathbf F$ и циркуляции выводятся **аналитически** в отчёте. Ниже — только иллюстрация поля для одной фиксированной опорной точки.

In [ ]:
waypoints=choose_waypoints(path)
T=waypoints[min(len(waypoints)-1, max(1, len(waypoints)//2))]
x0,y0=key["x"],key["y"]
gxs=np.linspace(max(0,x0-2.0),min(WORKSPACE_W,x0+2.0),17)
gys=np.linspace(max(0,y0-2.0),min(WORKSPACE_H,y0+2.0),17)
X,Y=np.meshgrid(gxs,gys); U=np.zeros_like(X); V=np.zeros_like(Y)
for iy in range(Y.shape[0]):
    for ix in range(X.shape[1]):
        fx,fy=field((X[iy,ix],Y[iy,ix]),T,obs,scene["field_k"],scene["beta_s"],omega,key)
        n=max(1e-9,(fx*fx+fy*fy)**0.5); U[iy,ix]=fx/n; V[iy,ix]=fy/n
fig,ax=plt.subplots(figsize=(7,6)); ax.quiver(X,Y,U,V)
ax.scatter(*T,marker="*",s=120,label="фиксированная T")
ax.add_patch(Circle((x0,y0),model_forbidden_radius(key),fill=False,ls="--"))
ax.set(xlabel="x, м",ylabel="y, м",title="Локальное поле для фиксированной опорной точки")
ax.set_aspect("equal"); ax.legend(); plt.show()

## 4. Численная траектория
Сравните $\omega=0$ и штатное $\omega$. Не предполагается заранее, что один режим обязан быть лучше по всем метрикам.

In [ ]:
sim0=simulate(start,goal,path,obs,scene["field_k"],scene["beta_s"],0.0,key)
sim=simulate(start,goal,path,obs,scene["field_k"],scene["beta_s"],omega,key)
for name,d in [("omega=0",sim0),("штатное omega",sim)]:
    print(name,"reached=",d["reached"],"steps=",d["steps"],"d_min=",d["min_clearance"],"length=",d["trajectory_length"])

In [ ]:
fig,ax=plt.subplots(figsize=(10,6))
for o in obs:
    ax.add_patch(Circle((o["x"],o["y"]),model_forbidden_radius(o),fill=False,ls="--"))
P=np.array(path); ax.plot(P[:,0],P[:,1],"--",lw=1.5,label="A*")
for d,label in [(sim0,"omega=0"),(sim,"штатное omega")]:
    Q=np.array(d["trajectory"]); ax.plot(Q[:,0],Q[:,1],lw=2,label=label)
ax.scatter(*start,label="S"); ax.scatter(*goal,marker="*",s=100,label="G")
ax.set(xlim=(0,WORKSPACE_W),ylim=(0,WORKSPACE_H),xlabel="x, м",ylabel="y, м")
ax.set_aspect("equal"); ax.legend(); plt.show()

## Перед сдачей
- Не копируйте служебный код в основной текст отчёта.
- В отчёте важны **формула → вычисление → интерпретация**.
- A* и параметры симулятора считаются готовой инфраструктурой.
- Для поля фиксируйте одну опорную точку $T$; траекторию при переключаемой $T(t)$ показывайте отдельно.